# 3. BERT-based Transformer Fine-tuning
**Dataset:** CLINC150 — 151-class intent classification

**Approach:** Fine-tune DistilBERT on raw text, no manual preprocessing needed

**Expected Accuracy:** ~95–97%

**Requirements:** `pip install transformers datasets torch`

> GPU strongly recommended. On CPU training will be very slow — consider Google Colab (Runtime → T4 GPU).

### Imports & Device Setup

In [1]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir                  = './mobilebert_clinc',
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 3e-5,
    warmup_ratio                = 0.1,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'accuracy',
    fp16                        = False,
    report_to                   = 'none',
    seed                        = 42
)

# NOTE: EarlyStoppingCallback removed — it breaks evaluate() if called
# after a kernel restart without re-running train(). 3 epochs is short
# enough that early stopping is not necessary.
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset_torch,
    eval_dataset    = val_dataset_torch,
    compute_metrics = compute_metrics,
)


Using device: cpu


### Load Dataset

In [2]:
test_dataset = load_dataset('parquet', data_files={'test':       os.path.join('..', 'data', 'test-00000-of-00001.parquet')})
train_dataset = load_dataset('parquet', data_files={'train':      os.path.join('..', 'data', 'train-00000-of-00001.parquet')})
val_dataset = load_dataset('parquet', data_files={'validation': os.path.join('..', 'data', 'validation-00000-of-00001.parquet')})

train_df = train_dataset['train'].to_pandas()
val_df   = val_dataset['validation'].to_pandas()
test_df  = test_dataset['test'].to_pandas()

NUM_CLASSES = train_df['intent'].nunique()
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} | Classes: {NUM_CLASSES}')

Train: 10625 | Val: 3100 | Test: 5500 | Classes: 151


### Tokenizer & Dataset Class
DistilBERT is used here — it is 40% smaller and 60% faster than BERT while retaining ~97% of BERT's performance.

In [3]:
MODEL_NAME = 'google/mobilebert-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt'
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset_torch = IntentDataset(train_df['text'], train_df['intent'], tokenizer)
val_dataset_torch = IntentDataset(val_df['text'],   val_df['intent'],   tokenizer)
test_dataset_torch  = IntentDataset(test_df['text'],  test_df['intent'],  tokenizer)

print(f'Sample encoding keys: {list(train_dataset_torch[0].keys())}')

config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

e:\Github Project\Intent-Router-Engine\.venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\TPY\.cache\huggingface\hub\models--google--mobilebert-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Sample encoding keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']


### Load Pre-trained Model

In [4]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES
).to(DEVICE)

# # Add this right after loading the model:
# # Freeze all layers except the final classifier
# for name, param in model.named_parameters():
#     if 'classifier' not in name:
#         param.requires_grad = False

# # Verify: should show only ~150k trainable params instead of 66M
# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f'Trainable params: {trainable:,}')

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,} | Trainable: {trainable_params:,}')

pytorch_model.bin:   0%|          | 0.00/147M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1111 [00:00<?, ?it/s]

MobileBertForSequenceClassification LOAD REPORT from: google/mobilebert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.dense.weight               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
mobilebert.embeddings.position_ids         | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignore

Total params: 24,659,351 | Trainable: 24,659,351


model.safetensors:   0%|          | 0.00/147M [00:00<?, ?B/s]

### Training Configuration

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir                  = './mobilebert_clinc',
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 3e-5,
    warmup_ratio                = 0.1,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'accuracy',
    fp16                        = False,
    report_to                   = 'none',
    seed                        = 42
)

# NOTE: EarlyStoppingCallback removed — it breaks evaluate() if called
# after a kernel restart without re-running train(). 3 epochs is short
# enough that early stopping is not necessary.
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset_torch,
    eval_dataset    = val_dataset_torch,
    compute_metrics = compute_metrics,
)


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

### Train

In [6]:
trainer.train()

e:\Github Project\Intent-Router-Engine\.venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,3.594350,3.473143,0.527419


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

e:\Github Project\Intent-Router-Engine\.venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

### Evaluation

In [ ]:
# --- Evaluation ---
# If you restarted the kernel, re-run all cells from the top first.
# The best checkpoint is loaded automatically via load_best_model_at_end=True.

val_results  = trainer.evaluate(val_dataset_torch,  metric_key_prefix='val')
test_results = trainer.evaluate(test_dataset_torch, metric_key_prefix='test')

print(f'Validation Accuracy : {val_results["val_accuracy"]*100:.2f}%')
print(f'Test Accuracy       : {test_results["test_accuracy"]*100:.2f}%')


In [ ]:
# Detailed per-class report
test_preds  = trainer.predict(test_dataset_torch)
y_test_pred = np.argmax(test_preds.predictions, axis=-1)
y_test      = test_df['intent'].tolist()

print(classification_report(y_test, y_test_pred))

### Inference Helper

In [ ]:
intent_mapping = {
    0: 'restaurant_reviews', 1: 'nutrition_info', 2: 'account_blocked', 3: 'oil_change_how', 4: 'time',
    5: 'weather', 6: 'redeem_rewards', 7: 'interest_rate', 8: 'gas_type', 9: 'accept_reservations',
    10: 'smart_home', 11: 'user_name', 12: 'report_lost_card', 13: 'repeat', 14: 'whisper_mode',
    # ... (full mapping omitted for brevity — paste full dict from original notebook)
    82: 'greeting', 83: 'alarm', 114: 'goodbye'
}

def predict_intent(text: str) -> str:
    model.eval()
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=64, padding=True).to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits
    pred_id = torch.argmax(logits, dim=-1).item()
    return intent_mapping.get(pred_id, str(pred_id))

# Quick test
for sample in ['what is the weather today', 'book a flight to London', 'tell me a joke']:
    print(f'{sample!r:45s} → {predict_intent(sample)}')